# **Detecção de Ocupação com Sensores IoT**
## Pipeline Preditivo aplicado à Automação Predial e Eficiência Energética na Indústria 4.0

**Projeto Avaliativo:** Integração de IA com Dados (SCTEC Aperfeiçoar)

**Aluna:** Anaysa Pereira Lopes

## Desafio

Um complexo industrial dotado de sensores ambientais IoT necessita saber, em tempo real, se suas salas técnicas e escritórios estão ocupados ou desocupados. Com essa informação, o ar-condicionado e a iluminação podem ser desligados automaticamente quando não há ninguém presente, diminuindo o consumo de energia.

A finalidade deste projeto é criar um modelo capaz de realizar essa previsão com base na leitura dos sensores.

A base usada é a [Occupancy Detection Dataset](https://archive.ics.uci.edu/dataset/357/occupancy+detection) (UCI Machine Learning Repository), com as seguintes variáveis:

* Temperature: temperatura do ambiente (°C)
* Humidity: umidade relativa do ar (%)
* Light: luminosidade (Lux)
* CO2: concentração de dióxido de carbono (ppm)
* HumidityRatio: razão de umidade, calculada a partir da temperatura e da umidade
* Occupancy: variável alvo (0 = desocupado, 1 = ocupado)

In [ ]:
# Bibliotecas

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Configurações visuais dos gráficos
sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams["figure.figsize"] = (10, 5)

## Base de Dados

A base foi fornecida junto com o enunciado do projeto e fica na pasta `data/`. Usei caminho relativo na leitura para que o notebook funcione em qualquer computador.

In [ ]:
# Carregamento dos dados

df = pd.read_csv("data/occupancy_completo.csv")
df.head()

## 1. Análise Exploratória (EDA)

In [ ]:
# Quantidade de linhas e colunas
df.shape

In [ ]:
# Tipo de cada coluna e valores preenchidos
df.info()

In [ ]:
# Amostra aleatória
df.sample(20, random_state=42)

### 1.1 Ajuste de formato

Como alguns valores dos sensores apresentavam o ponto decimal deslocado (por exemplo, `272.675` em vez de `27.2675`), as colunas foram interpretadas como texto. Para corrigir, retirei os pontos e ajustei a posição do decimal de acordo com a faixa de valores de cada sensor.

In [ ]:
# Correção de Temperature e Humidity
for coluna in ["Temperature", "Humidity"]:
    digitos = df[coluna].str.replace(".", "", regex=False)
    df[coluna] = (digitos.str[:2] + "." + digitos.str[2:]).astype(float)

df[["Temperature", "Humidity"]].describe()

In [ ]:
# Correção de HumidityRatio
digitos = df["HumidityRatio"].str.replace(".", "", regex=False)
df["HumidityRatio"] = ("0.00" + digitos).astype(float)

df["HumidityRatio"].describe()

No CO2, só existe uma posição possível para o decimal dentro da faixa da base (412 a 2.076 ppm), então os valores foram corrigidos. Na Light, mais de uma posição é possível, então os valores incorretos foram transformados em NaN e serão tratados na etapa de limpeza.

In [ ]:
# Correção de CO2
def corrigir_co2(valor):
    if valor.count(".") <= 1:
        return float(valor)
    digitos = valor.replace(".", "")
    if int(digitos[:3]) >= 412:
        return float(digitos[:3] + "." + digitos[3:])
    return float(digitos[:4] + "." + digitos[4:])

df["CO2"] = df["CO2"].apply(corrigir_co2)
df["CO2"].describe()

In [ ]:
# Conversão de Light
df["Light"] = pd.to_numeric(df["Light"], errors="coerce")

df.info()

A coluna `date` está no formato de texto e simplesmente armazena o instante de cada medição. Como o objetivo é identificar a ocupação por meio das leituras dos sensores, essa coluna foi eliminada.

In [ ]:
# Remoção da coluna date
df = df.drop(columns=["date"])
df.head()

### 1.2 Resumo estatístico e gráficos

In [ ]:
# Resumo estatístico
df.describe()

In [ ]:
# Proporção das classes de Occupancy
ax = sns.countplot(data=df, x="Occupancy", hue="Occupancy", legend=False)
ax.set_title("Distribuição da variável alvo")
ax.set_xlabel("Occupancy (0 = desocupado, 1 = ocupado)")
ax.set_ylabel("Quantidade de registros")
plt.show()

df["Occupancy"].value_counts(normalize=True)

In [ ]:
# Distribuição das variáveis preditoras
preditoras = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, coluna in zip(axes.flat, preditoras):
    sns.histplot(data=df, x=coluna, hue="Occupancy", bins=30, ax=ax)
    ax.set_title(coluna)

axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Correlação de Pearson entre as variáveis
cores = sns.color_palette("deep")
paleta_calor = sns.blend_palette([cores[0], "white", cores[1]], as_cmap=True)

plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(method="pearson"), annot=True, fmt=".2f", cmap=paleta_calor, vmin=-1, vmax=1)
plt.title("Correlação de Pearson entre as variáveis")
plt.show()

In [ ]:
# Relação entre luminosidade, CO2 e ocupação
ax = sns.scatterplot(data=df, x="Light", y="CO2", hue="Occupancy", alpha=0.6, s=15)
ax.set_title("Relação entre luminosidade, CO2 e ocupação")
plt.show()

In [ ]:
# Redundância entre Humidity e HumidityRatio
ax = sns.scatterplot(data=df, x="Humidity", y="HumidityRatio", alpha=0.6, s=15)
ax.set_title("Humidity x HumidityRatio")
plt.show()

### 1.3 Análise exploratória

O conjunto de dados conta com 20.560 registros, sendo que a sala aparece ocupada em apenas 23% das amostras. Devido a esse desbalanceamento entre as classes, a separação entre treino e teste deve utilizar estratificação (`stratify`) para preservar a mesma proporção em ambos os subconjuntos.

Na matriz de correlação, `Light` apresenta associação muito forte com a ocupação (0,92): permanece próxima de zero quando o ambiente está vazio e sobe para cerca de 400 a 500 lux quando há pessoas. `Temperature` (0,56) e `CO2` (0,50) exibem correlação moderada. O nível de CO2 aumenta com a presença de pessoas, mas cai lentamente; no gráfico de dispersão, isso se reflete em observações com iluminação apagada e concentrações residuais de CO2 ainda elevadas. Já `Humidity` (0,05) e `HumidityRatio` (0,26) têm correlação fraca com a variável alvo.

Entre si, `Humidity` e `HumidityRatio` registram correlação de 0,93, sinalizando redundância que pode distorcer o cálculo de distâncias no KNN. A `HumidityRatio` captura melhor a relação com a ocupação por expressar a massa real de vapor d'água no ar, mas optei por manter as duas nesta etapa.

As variáveis também operam em escalas bastante distintas: a `HumidityRatio` varia na ordem de milésimos, enquanto `Light` e `CO2` alcançam valores na casa dos milhares. Essa diferença não impacta modelos baseados em árvores, como a Árvore de Decisão, mas afeta algoritmos baseados em distância, como o KNN. Por isso, o pipeline do KNN aplicará `StandardScaler` antes do treinamento.
